In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
# Force system to expose both GPUs (0 and 1) to PyTorch
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import torch
print("GPUs available:", torch.cuda.device_count()) 
# This should now output: GPUs available: 2

In [ ]:
import yaml, os

DATA_ROOT = "/kaggle/input/datasets/towfiqurrashid/dhaka-cc-dataset"  # <-- adjust to your path

cfg = {
    "path":  DATA_ROOT,
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    9,
    "names": ["rickshaw","car","bike","bus","cng","bicycle","truck","van","leguna"],
}
with open("/kaggle/working/data.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(os.listdir(DATA_ROOT))   # sanity check

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    # print folders that contain an 'images' dir (i.e. the split folders)
    if "images" in dirs:
        print(root)

In [ ]:
import yaml, os

# <-- set this to whatever step 1 printed (the parent of train/valid/test)
DATA_ROOT = "/kaggle/input/datasets/towfiqurrashid/dhaka-cc-dataset/merged_dataset"

# sanity check before training
for s in ["train", "valid", "test"]:
    p = os.path.join(DATA_ROOT, s, "images")
    print(p, "->", "OK" if os.path.isdir(p) else "MISSING")

cfg = {
    "path":  DATA_ROOT,
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    9,
    "names": ["rickshaw","car","bike","bus","cng","bicycle","truck","van","leguna"],
}
with open("/kaggle/working/data.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("wrote /kaggle/working/data.yaml")

In [ ]:
!pip install ultralytics -q

In [ ]:
import torch; print("GPUs:", torch.cuda.device_count())   # should print 2

In [ ]:
!nvidia-smi

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11m.pt")          # m for accuracy; use s if you want even faster

model.train(
    data="/kaggle/working/data.yaml",
    imgsz=960,            # was 1280 — big speedup, keeps small-object gains
    device=[0,1],        # USE BOTH GPUs
    batch=16,             # explicit (required for multi-GPU); 8/GPU
    epochs=100,
    patience=25,
    cos_lr=True,
    close_mosaic=15,
    workers=8,
    project="/kaggle/working/runs",
    name="yolo11m_960_2gpu",
    save_period=10,       # checkpoints survive a crash/timeout
)